# URL Phishing Detection

RF + XGBoost + GB soft-voting ensemble. Augments the legit URLs to fix a path/www/HTTP bias in the original dataset before training.

## 1. Imports

In [ ]:
import re
import math
import random
from collections import Counter
from urllib.parse import urlparse

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

random.seed(42)


## 2. Load dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = '/content/drive/MyDrive/PhiUSIIL_Phishing_URL_Dataset.csv'

df = pd.read_csv(DATASET_PATH)
print(df.shape, df['label'].value_counts().to_dict())


## 3. Augment legit URLs

Original legit URLs are almost all `https://www...` with no path, so the model was learning "has a path" or "not HTTPS" as phishing signals instead of anything real. Adding path/www/http variants plus some real-world anchors fixes that.

In [ ]:
PATH_TEMPLATES = [
    "/about", "/about-us", "/contact", "/contact-us", "/products", "/services",
    "/blog", "/blog/2024-annual-report", "/news/latest-updates", "/user/profile",
    "/login", "/signin", "/account/settings", "/search?q=information",
    "/category/electronics", "/docs/getting-started", "/api/v1/users",
    "/help/faq", "/support", "/careers", "/pricing", "/team", "/privacy-policy",
    "/terms-of-service", "/download", "/shop/item-12345", "/article/how-to-guide",
    "/dashboard", "/settings/profile", "/checkout", "/cart",
    "/questions/tagged/python", "/wiki/Machine_learning", "/wiki/Artificial_intelligence",
]

CURATED_LEGIT_ANCHORS = [
    "https://github.com/anthropics", "https://github.com/torvalds/linux",
    "https://en.wikipedia.org/wiki/Phishing", "https://en.wikipedia.org/wiki/Machine_learning",
    "https://www.wikipedia.org/wiki/Artificial_intelligence",
    "https://stackoverflow.com/questions/12345/how-to-fix-error",
    "https://stackoverflow.com/questions/tagged/python",
    "https://stackoverflow.com/questions/tagged/javascript",
    "https://www.amazon.com/dp/B08N5WRWNW", "https://www.amazon.com/gp/product/B01N5IB20Q",
    "https://www.nytimes.com/2024/05/12/technology/ai-news.html",
    "https://www.bbc.com/news/world-europe-12345678",
    "https://www.reddit.com/r/programming/comments/abc123/title",
    "https://medium.com/@author/article-title-123abc",
    "https://docs.python.org/3/library/functions.html",
    "https://developer.mozilla.org/en-US/docs/Web/JavaScript",
    "https://www.linkedin.com/in/some-profile-name",
    "https://twitter.com/username/status/1234567890",
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ",
    "https://news.ycombinator.com/item?id=12345678",
    "https://www.imdb.com/title/tt0111161/",
    "https://www.npmjs.com/package/react",
    "https://pypi.org/project/numpy/",
    "https://www.coursera.org/learn/machine-learning",
    "https://www.udemy.com/course/python-for-beginners/",
    "https://www.researchgate.net/publication/12345",
    "https://scholar.google.com/citations?user=abc123",
    "https://www.investopedia.com/terms/p/phishing.asp",
    "https://www.cnn.com/2024/05/12/tech/article-title",
    "https://www.theguardian.com/technology/2024/may/12/article",
    "https://www.forbes.com/sites/author/2024/05/12/article-title/",
    "https://www.microsoft.com/en-us/microsoft-365/support",
    "https://cloud.google.com/docs/tutorials",
    "https://aws.amazon.com/getting-started/",
    "https://www.apple.com/shop/buy-iphone",
    "https://support.google.com/accounts/answer/12345",
    "https://help.netflix.com/en/node/12345",
    "https://www.ebay.com/itm/12345678901",
    "https://www.etsy.com/listing/123456789/product-name",
    "https://www.walmart.com/ip/product-name/123456",
    "https://www.target.com/p/product-name/-/A-12345678",
    "https://www.bestbuy.com/site/product-name/1234567.p",
    "https://www.wsj.com/articles/some-article-title-12345",
    "https://techcrunch.com/2024/05/12/article-title/",
    "https://www.wired.com/story/article-title/",
    "https://arxiv.org/abs/2401.12345",
    "https://www.nature.com/articles/s41586-024-12345-6",
    "http://example.com", "http://example.org", "http://example.net",
    "http://neverssl.com", "http://info.cern.ch",
]

def strip_www(url):
    return url.replace("://www.", "://", 1)

def to_http(url):
    return url.replace("https://", "http://", 1) if url.startswith("https://") else url

legit_urls = df.loc[df['label'] == 1, 'URL'].tolist()
sampled = random.sample(legit_urls, min(40000, len(legit_urls)))

augmented_rows = []
for url in sampled:
    path = random.choice(PATH_TEMPLATES)
    augmented_rows.append({'URL': url.rstrip('/') + path, 'label': 1})
    augmented_rows.append({'URL': strip_www(url), 'label': 1})
    augmented_rows.append({'URL': strip_www(url).rstrip('/') + path, 'label': 1})

# plain-HTTP versions too, just enough that HTTPS stops being a near-perfect signal
http_sampled = random.sample(legit_urls, min(15000, len(legit_urls)))
augmented_rows += [{'URL': to_http(url), 'label': 1} for url in http_sampled]

# repeat the curated anchors so the tree models actually pick up on them
for url in CURATED_LEGIT_ANCHORS:
    augmented_rows += [{'URL': url, 'label': 1}] * 20

augmented_df = pd.DataFrame(augmented_rows)
df_aug = pd.concat([df[['URL', 'label']], augmented_df], ignore_index=True)
print(df_aug.shape, df_aug['label'].value_counts().to_dict())


## 4. Feature extraction

Same function gets reused for training and for inference in the API later.

In [ ]:
SUSPICIOUS_KEYWORDS = ['login', 'verify', 'secure', 'account', 'update',
                       'confirm', 'banking', 'signin', 'password', 'suspend']
SHORTENERS = ['bit.ly', 'tinyurl', 'goo.gl', 't.co', 'ow.ly', 'is.gd', 'buff.ly']

FEATURE_ORDER = [
    'URLLength', 'DomainLength', 'IsDomainIP', 'TLDLength', 'NoOfSubDomain',
    'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio',
    'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL',
    'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL',
    'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS',
    'HasIPPattern', 'HasAtSymbol', 'HyphenCount', 'DotCount',
    'HasSuspiciousKeyword', 'URLEntropy', 'UsesShortener',
    'PathLength', 'HasPath'
]

def extract_features(url: str) -> dict:
    url = str(url)
    parsed = urlparse(url if '://' in url else 'http://' + url)
    domain = parsed.netloc
    tld = domain.split('.')[-1] if '.' in domain else ''

    f = {}
    f['URLLength'] = len(url)
    f['DomainLength'] = len(domain)
    f['IsDomainIP'] = 1 if re.match(r'^(\d{1,3}\.){3}\d{1,3}$', domain) else 0
    f['TLDLength'] = len(tld)
    f['NoOfSubDomain'] = max(domain.count('.') - 1, 0)
    f['HasObfuscation'] = 1 if '%' in url else 0
    f['NoOfObfuscatedChar'] = url.count('%')
    f['ObfuscationRatio'] = url.count('%') / len(url) if len(url) > 0 else 0
    f['NoOfLettersInURL'] = sum(c.isalpha() for c in url)
    f['LetterRatioInURL'] = f['NoOfLettersInURL'] / len(url) if len(url) > 0 else 0
    f['NoOfDegitsInURL'] = sum(c.isdigit() for c in url)
    f['DegitRatioInURL'] = f['NoOfDegitsInURL'] / len(url) if len(url) > 0 else 0
    f['NoOfEqualsInURL'] = url.count('=')
    f['NoOfQMarkInURL'] = url.count('?')
    f['NoOfAmpersandInURL'] = url.count('&')
    special_chars = sum(1 for c in url if not c.isalnum() and c not in ['.', '/', ':'])
    f['NoOfOtherSpecialCharsInURL'] = special_chars
    f['SpacialCharRatioInURL'] = special_chars / len(url) if len(url) > 0 else 0
    f['IsHTTPS'] = 1 if url.startswith('https://') else 0
    f['HasIPPattern'] = 1 if re.search(r'(\d{1,3}\.){3}\d{1,3}', url) else 0
    f['HasAtSymbol'] = 1 if '@' in url else 0
    f['HyphenCount'] = url.count('-')
    f['DotCount'] = url.count('.')
    f['HasSuspiciousKeyword'] = 1 if any(k in url.lower() for k in SUSPICIOUS_KEYWORDS) else 0
    counts = Counter(url)
    probs = [c / len(url) for c in counts.values()] if len(url) > 0 else [0]
    f['URLEntropy'] = -sum(p * math.log2(p) for p in probs if p > 0)
    f['UsesShortener'] = 1 if any(s in url.lower() for s in SHORTENERS) else 0
    path = parsed.path
    f['PathLength'] = len(path)
    f['HasPath'] = 1 if path not in ('', '/') else 0

    return f

extract_features("http://neverssl.com")


## 5. Build feature matrix

In [ ]:
feature_rows = df_aug['URL'].apply(extract_features)
X = pd.DataFrame(list(feature_rows))[FEATURE_ORDER]
y = df_aug['label']
X.shape


## 6. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape


## 7. Train individual models

In [ ]:
individual_results = {}

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
individual_results['Random Forest'] = {
    'accuracy': accuracy_score(y_test, rf_preds),
    'precision': precision_score(y_test, rf_preds),
    'recall': recall_score(y_test, rf_preds),
    'f1': f1_score(y_test, rf_preds),
}

xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                     random_state=42, eval_metric='logloss', n_jobs=-1)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
individual_results['XGBoost'] = {
    'accuracy': accuracy_score(y_test, xgb_preds),
    'precision': precision_score(y_test, xgb_preds),
    'recall': recall_score(y_test, xgb_preds),
    'f1': f1_score(y_test, xgb_preds),
}

gb = GradientBoostingClassifier(n_estimators=150, max_depth=5, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
gb_preds = gb.predict(X_test)
individual_results['Gradient Boosting'] = {
    'accuracy': accuracy_score(y_test, gb_preds),
    'precision': precision_score(y_test, gb_preds),
    'recall': recall_score(y_test, gb_preds),
    'f1': f1_score(y_test, gb_preds),
}

pd.DataFrame(individual_results).T


## 8. Hybrid model (soft voting)

In [ ]:
hybrid_model = VotingClassifier(
    estimators=[('rf', rf), ('xgb', xgb), ('gb', gb)],
    voting='soft',
    n_jobs=-1
)
hybrid_model.fit(X_train, y_train)
hybrid_preds = hybrid_model.predict(X_test)

hybrid_results = {
    'accuracy': accuracy_score(y_test, hybrid_preds),
    'precision': precision_score(y_test, hybrid_preds),
    'recall': recall_score(y_test, hybrid_preds),
    'f1': f1_score(y_test, hybrid_preds),
}

all_results = pd.DataFrame(individual_results).T
all_results.loc['Hybrid'] = hybrid_results
print(all_results)
print(classification_report(y_test, hybrid_preds, target_names=['Phishing', 'Legitimate']))


## 9. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, hybrid_preds)
ConfusionMatrixDisplay(cm, display_labels=['Phishing', 'Legitimate']).plot(cmap='Blues', values_format='d')
plt.title("Hybrid Model - Confusion Matrix")
plt.show()

print(f"False negatives: {cm[0][1]}, false positives: {cm[1][0]}")


## 10. Feature importance

In [ ]:
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 8))
plt.title("Feature Importance (Random Forest)")
plt.barh(range(len(importances)), importances[indices][::-1])
plt.yticks(range(len(importances)), [FEATURE_ORDER[i] for i in indices][::-1])
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 11. Sanity check on real URLs

In [ ]:
test_urls = [
    ("https://www.google.com", "legit"),
    ("https://www.wikipedia.org", "legit"),
    ("https://github.com/anthropics", "legit, has path, no www"),
    ("https://www.wikipedia.org/wiki/Artificial_intelligence", "legit, previously failed"),
    ("https://www.amazon.com/dp/B08N5WRWNW", "legit, has path"),
    ("https://stackoverflow.com/questions/tagged/python", "legit, previously failed"),
    ("https://www.reuters.com/technology/some-article-2024", "legit, unseen domain"),
    ("http://example.com", "legit, plain HTTP"),
    ("http://example.org", "legit, plain HTTP"),
    ("http://neverssl.com", "legit, plain HTTP"),
    ("http://192.168.1.1/login-verify-account.php", "phishing, IP + keywords"),
    ("https://paypal-secure-login.suspicious-domain.tk", "phishing, brand impersonation"),
    ("http://bit.ly/free-prize-claim-now", "phishing, shortener"),
]

for url, note in test_urls:
    X_new = pd.DataFrame([extract_features(url)])[FEATURE_ORDER]
    pred = hybrid_model.predict(X_new)[0]
    proba = hybrid_model.predict_proba(X_new)[0]
    label = "Legitimate" if pred == 1 else "Phishing"
    print(f"{url} -> {label} ({max(proba):.0%})  [{note}]")


## 12. Export

Saved as three separate files instead of one combined pickle, since XGBoost's pickle format doesn't travel well across platforms.

In [ ]:
rf_fitted = hybrid_model.named_estimators_['rf']
xgb_fitted = hybrid_model.named_estimators_['xgb']
gb_fitted = hybrid_model.named_estimators_['gb']

joblib.dump(rf_fitted, 'url_rf_model.pkl')
joblib.dump(gb_fitted, 'url_gb_model.pkl')
xgb_fitted.save_model('url_xgb_model.json')

from google.colab import files
files.download('url_rf_model.pkl')
files.download('url_gb_model.pkl')
files.download('url_xgb_model.json')
